# Qwen3-1.7B safety post-training on one Kaggle T4
This notebook orchestrates the repository scripts. Candidate acquisition is separate from human approval, and real training is opt-in.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-c', "import torch; print('torch', torch.__version__, 'CUDA', torch.version.cuda); print('GPU', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')"], check=True)

## Repository and compatible dependencies
Clone/pull the private repository into `/kaggle/working`, enable Internet and a T4 GPU, and set `REPO` below. The requirements deliberately leave Kaggle's PyTorch/CUDA installation alone.

In [ ]:
from pathlib import Path
REPO = Path('/kaggle/working/black-box-diff-exploration')
assert (REPO / 'train.py').exists(), 'Clone/copy the repository into /kaggle/working and update REPO'
%cd {REPO}
%env HF_HOME=/kaggle/working/hf_cache

In [ ]:
%pip install --upgrade -q -r requirements-kaggle.txt
!python scripts/verify_dependencies.py

If verification still reports an old imported package, restart the Kaggle kernel once, rerun the repository cell, and rerun the verifier.

## Fixture preflight and one-step smoke test
This readiness check uses bundled synthetic fixtures, so it does not expect the real study data yet. Smoke artifacts are isolated under the `_smoke` namespace.

In [ ]:
!python scripts/kaggle_preflight.py --smoke-test
!python run_training_matrix.py --conditions M1 M2 M3 --seeds 42 --smoke-test
!python evaluate_safety.py --condition M1 --seed 42 --smoke-test
!python evaluate_safety.py --condition M2 --seed 42 --smoke-test
!python evaluate_safety.py --condition M3 --seed 42 --smoke-test

## Acquire public candidates (not approved study data)
Running the next command downloads revision-pinned candidates from UltraChat 200k, PKU-SafeRLHF, HarmBench, and XSTest. It does not create train/evaluation files. Review source licenses first; PKU-SafeRLHF is non-commercial.

In [ ]:
# Run once. It refuses to overwrite an existing review pool.
# !python scripts/acquire_public_data.py --train-examples 300 --eval-examples 100
# !python scripts/review_status.py

Review every retained record in `data/review/*_candidates.jsonl` as documented in `data/README.md`. Set `review_status`, `final_category`, and—for safety candidates—`use`. Keep dual-use evaluation records out of training. Then run the next commands deliberately.

In [ ]:
# !python scripts/review_status.py
# !python scripts/finalize_reviewed_data.py
# Generate M3 targets with the teacher chosen in the preregistration:
# !python scripts/generate_constitutional_targets.py --teacher-model Qwen/Qwen3-1.7B
# !python scripts/freeze_evaluation.py
# !python scripts/validate_experiment.py
# !python scripts/check_experimental_balance.py --model-tokenizer
# !python scripts/kaggle_preflight.py

## Selected real training job
Set `RUN_SELECTED_JOB=True` only after the real-data preflight prints `READY`. Change the condition and seed deliberately.

In [ ]:
RUN_SELECTED_JOB = False
CONDITION = 'M2'
SEED = 42
CONFIGS = {'M1': 'configs/m1_benign.yaml', 'M2': 'configs/m2_safety_sft.yaml', 'M3': 'configs/m3_constitutional.yaml'}
assert CONDITION in CONFIGS and SEED in (42, 123, 456)
if RUN_SELECTED_JOB:
    subprocess.run([sys.executable, 'create_m0_manifest.py'], check=True)
    subprocess.run([sys.executable, 'train.py', '--config', CONFIGS[CONDITION], '--seed', str(SEED), '--resume'], check=True)
else:
    print('Real training is disabled. Complete review/preflight, then set RUN_SELECTED_JOB=True.')

## Immediate frozen audit
The built-in heuristic verifies the pipeline only. Configure a validated `SafetyScorer` for research conclusions.

In [ ]:
if RUN_SELECTED_JOB:
    subprocess.run([sys.executable, 'evaluate_safety.py', '--condition', CONDITION, '--seed', str(SEED)], check=True)

## Artifact summary and optional matrix

In [ ]:
!python run_training_matrix.py --conditions M1 M2 M3 --seeds 42 123 456 --dry-run
# Run the full matrix only when intended:
# !python run_training_matrix.py --conditions M1 M2 M3 --seeds 42 123 456 --resume
# Optional M4 after its matching M3:
# !python train_dpo.py --config configs/m4_dpo.yaml --seed 42 --resume